# No-Label RF Segmentation Head Workbench

Goal: train a segmentation head for **noise vs non-noise** without manually labeled diverse signal classes.

This notebook is designed as a restart point for:
- learning robust noise behavior from real/background data
- creating synthetic non-noise overlays for weak labels
- calibrating thresholds so pure-noise inputs map to all-noise masks

## Strategy Summary

1. Build a **noise-first dataset** from real captures (or noise-only windows).
2. Generate synthetic signal overlays (tones, chirps, bursts, hops, blocks) with known masks.
3. Freeze DINO backbone initially and train a lightweight binary segmentation head.
4. Include many pure-noise samples with all-zero masks.
5. Calibrate decision threshold on a held-out pure-noise set to enforce low false alarms.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 2026
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

In [ ]:
@dataclass
class Config:
    h: int = 256
    w: int = 256
    p_noise_only: float = 0.5
    snr_db_min: float = -25.0
    snr_db_max: float = 20.0
    batch_size: int = 6
    epochs: int = 8
    lr: float = 2e-4
    pure_noise_target_fpr: float = 0.01
    dinov3_location: str = '/home/sat3737/holoscan_demo_workspace/dinov3'
    dinov3_model_name: str = 'dinov3_vitb16'
    dinov3_weights: str = '/home/sat3737/holoscan_demo_workspace/dinov3_weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'

cfg = Config()
cfg

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device=device).view(1, 3, 1, 1)

if Path(cfg.dinov3_location).exists():
    dino_backbone = torch.hub.load(
        repo_or_dir=cfg.dinov3_location,
        model=cfg.dinov3_model_name,
        source='local',
        weights=cfg.dinov3_weights,
    )
else:
    dino_backbone = torch.hub.load(
        repo_or_dir='facebookresearch/dinov3',
        model=cfg.dinov3_model_name,
        source='github',
        weights=cfg.dinov3_weights,
    )

dino_backbone = dino_backbone.to(device).eval()
for param in dino_backbone.parameters():
    param.requires_grad = False

print(f'Loaded frozen DINO backbone: {cfg.dinov3_model_name} on {device}')

## Synthetic Overlay Primitives

These are weak-label generators. They do not require signal class labels and produce binary masks for training.

In [ ]:
def draw_tone(mask, rng):
    h, w = mask.shape
    y = int(rng.integers(0, h))
    thickness = int(rng.integers(1, 4))
    mask[max(0, y-thickness):min(h, y+thickness+1), :] = 1

def draw_chirp(mask, rng):
    h, w = mask.shape
    y0 = int(rng.integers(0, h))
    y1 = int(rng.integers(0, h))
    xs = np.arange(w)
    ys = (y0 + (y1 - y0) * (xs / max(1, w - 1))).astype(int)
    for x, y in zip(xs, ys):
        mask[max(0, y-1):min(h, y+2), x] = 1

def draw_burst(mask, rng):
    h, w = mask.shape
    bh = int(rng.integers(max(3, h//24), max(4, h//8)))
    bw = int(rng.integers(max(6, w//24), max(8, w//6)))
    y0 = int(rng.integers(0, max(1, h - bh)))
    x0 = int(rng.integers(0, max(1, w - bw)))
    mask[y0:y0+bh, x0:x0+bw] = 1

def draw_hops(mask, rng):
    hops = int(rng.integers(2, 7))
    for _ in range(hops):
        draw_burst(mask, rng)

PRIMITIVES = [draw_tone, draw_chirp, draw_burst, draw_hops]

def sample_signal_mask(h, w, rng):
    mask = np.zeros((h, w), dtype=np.uint8)
    n_obj = int(rng.integers(1, 4))
    for _ in range(n_obj):
        prim = PRIMITIVES[int(rng.integers(0, len(PRIMITIVES)))]
        prim(mask, rng)
    return mask

In [ ]:
def synth_background(h, w, rng):
    base = rng.normal(0.0, 1.0, size=(h, w)).astype(np.float32)
    row_bias = rng.normal(0, 0.15, size=(h, 1)).astype(np.float32)
    col_bias = rng.normal(0, 0.05, size=(1, w)).astype(np.float32)
    return base + row_bias + col_bias

def inject_signal(background, signal_mask, snr_db, rng):
    bg = background.astype(np.float32)
    sig = signal_mask.astype(np.float32)
    if sig.sum() == 0:
        return bg

    target_ratio = 10 ** (snr_db / 20.0)
    bg_rms = np.sqrt(np.mean(bg**2) + 1e-8)
    amp = bg_rms * target_ratio

    texture = rng.normal(0.7, 0.2, size=bg.shape).astype(np.float32)
    texture = np.clip(texture, 0.0, None)
    return bg + amp * texture * sig

## Dataset (Weak Labels)

Replace `synth_background` with real noise spectrogram crops for production.

In [ ]:
class WeakRFDataset(Dataset):
    def __init__(self, n_samples, cfg, rng):
        self.n_samples = n_samples
        self.cfg = cfg
        self.rng = rng

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        h, w = self.cfg.h, self.cfg.w
        bg = synth_background(h, w, self.rng)

        noise_only = self.rng.random() < self.cfg.p_noise_only
        if noise_only:
            mask = np.zeros((h, w), dtype=np.uint8)
            x = bg
        else:
            mask = sample_signal_mask(h, w, self.rng)
            snr_db = float(self.rng.uniform(self.cfg.snr_db_min, self.cfg.snr_db_max))
            x = inject_signal(bg, mask, snr_db, self.rng)

        x = (x - x.mean()) / (x.std() + 1e-6)
        x = torch.from_numpy(x[None, ...]).float()
        y = torch.from_numpy(mask[None, ...].astype(np.float32))
        return x, y

train_ds = WeakRFDataset(n_samples=512, cfg=cfg, rng=np.random.default_rng(SEED))
val_ds = WeakRFDataset(n_samples=128, cfg=cfg, rng=np.random.default_rng(SEED + 1))
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

In [ ]:
x, y = next(iter(train_loader))
fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    ax[0, i].imshow(x[i, 0].numpy(), cmap='viridis')
    ax[0, i].set_title('Input')
    ax[0, i].axis('off')
    ax[1, i].imshow(y[i, 0].numpy(), cmap='gray', vmin=0, vmax=1)
    ax[1, i].set_title('Weak mask')
    ax[1, i].axis('off')
plt.tight_layout()
plt.show()

## DINO-Feature Segmentation Head

This head now trains directly on **frozen DINO feature maps** extracted from each spectrogram.

In [ ]:
class DinoSegHead(nn.Module):
    def __init__(self, in_ch=768, hidden=192):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden // 2, 1, 1),
        )

    def forward(self, feat_map, out_hw):
        logits_small = self.decoder(feat_map)
        logits = F.interpolate(logits_small, size=out_hw, mode='bilinear', align_corners=False)
        return logits


@torch.no_grad()
def extract_dino_feat(batch_x):
    x = batch_x.to(device)  # [B,1,H,W], standardized
    x = x.repeat(1, 3, 1, 1)
    x = (x - x.amin(dim=(2, 3), keepdim=True)) / (x.amax(dim=(2, 3), keepdim=True) - x.amin(dim=(2, 3), keepdim=True) + 1e-6)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD

    feat = dino_backbone.get_intermediate_layers(x, n=1, reshape=True, norm=True)[0]
    return feat.float()


dino_ch = 768  # dinov3_vitb16 embed dim
seg_head = DinoSegHead(in_ch=dino_ch).to(device)
opt = torch.optim.AdamW(seg_head.parameters(), lr=cfg.lr)
bce = nn.BCEWithLogitsLoss()


def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

In [ ]:
def train_one_epoch(loader):
    seg_head.train()
    losses = []
    for x, y in loader:
        y = y.to(device)
        feat = extract_dino_feat(x)
        logits = seg_head(feat, out_hw=y.shape[-2:])
        loss = 0.7 * bce(logits, y) + 0.3 * dice_loss(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
    return np.mean(losses)


@torch.no_grad()
def eval_noise_fpr(loader, threshold=0.5):
    seg_head.eval()
    fp_pixels, total_noise_pixels = 0, 0
    for x, y in loader:
        y = y.to(device)
        feat = extract_dino_feat(x)
        logits = seg_head(feat, out_hw=y.shape[-2:])
        p = torch.sigmoid(logits)
        pred = (p >= threshold).float()
        noise_mask = (y == 0)
        fp_pixels += int((pred[noise_mask] > 0).sum().item())
        total_noise_pixels += int(noise_mask.sum().item())
    return fp_pixels / max(1, total_noise_pixels)


for epoch in range(cfg.epochs):
    tr = train_one_epoch(train_loader)
    fpr = eval_noise_fpr(val_loader, threshold=0.5)
    print(f'Epoch {epoch+1:02d} | loss={tr:.4f} | noise-FPR@0.5={fpr:.5f}')

## Threshold Calibration for Pure-Noise Acceptance

Choose the highest threshold that keeps pure-noise false alarm below your target.

In [ ]:
thresholds = np.linspace(0.1, 0.95, 18)
fprs = [eval_noise_fpr(val_loader, t) for t in thresholds]

valid = [t for t, fpr in zip(thresholds, fprs) if fpr <= cfg.pure_noise_target_fpr]
best_t = max(valid) if valid else 0.5
print(f'Chosen threshold={best_t:.3f} for target pure-noise FPR<={cfg.pure_noise_target_fpr:.4f}')

plt.figure(figsize=(6,4))
plt.plot(thresholds, fprs, marker='o')
plt.axhline(cfg.pure_noise_target_fpr, color='r', linestyle='--', label='target FPR')
plt.xlabel('Threshold')
plt.ylabel('Pure-noise false positive rate')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Next Steps for Your Main Pipeline

- Replace synthetic background with real noise-only spectrogram crops from your captures.
- Keep a dedicated pure-noise validation split and monitor pixel-FPR + frame-level false alarms.
- Add weak consistency loss across augmentations to stabilize low-SNR behavior.
- Optionally unfreeze the last DINO block after head convergence for domain adaptation.